# Aggregate 질문: 울타리에 몇 번 올라탔나요?

행동 횟수는 검색 상위 프레임 수나 `repeat_count`가 아니라, 질문에 맞는 행동 이벤트를 전체 기간에서 찾아 겹치는 구간을 정리해 계산합니다. 이 노트북은 `dog_escape` 영상의 울타리 올라타기를 발표 사례로 사용합니다.


In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))
from pipeline.search.aggregate_query import parse_count_action, event_matches_action, count_episodes

question = '강아지가 울타리에 몇 번 올라탔어?'
events = [
    {'id': 1, 'video_id': 'dog_escape', 'start_time': 6, 'end_time': 11, 'action': '강아지가 울타리에 올라탐', 'repeat_count': 8},
    {'id': 2, 'video_id': 'dog_escape', 'start_time': 8, 'end_time': 13, 'action': '강아지가 울타리에 올라탐', 'repeat_count': 3},
    {'id': 3, 'video_id': 'dog_escape', 'start_time': 25, 'end_time': 31, 'action': '강아지가 울타리에 올라탐', 'repeat_count': 5},
    {'id': 4, 'video_id': 'dog_escape', 'start_time': 32, 'end_time': 34, 'action': '강아지가 울타리 근처에 있음', 'repeat_count': 1},
]
action, label = parse_count_action(question)
matched = [event for event in events if event_matches_action(event, action)]
episodes = count_episodes(matched)
print('질문:', question)
print('행동 후보 ID:', [event['id'] for event in matched])
print('정리된 구간:', [(e['start_time'], e['end_time'], e['event_ids']) for e in episodes])
print('확인된 후보 구간 수:', len(episodes))

위의 **가상 이벤트**에서는 1·2번 기록이 겹쳐 한 구간이고 3번은 별도 구간입니다. 4번은 울타리 근처에만 있어 제외합니다. 따라서 후보 구간은 2개입니다. 이 숫자는 실제 영상의 검증된 정답이 아닙니다.

## 실제 DB 이벤트 조회

DB가 연결된 실행 환경에서 아래 `demo_video_id`를 설정해 실제 이벤트 후보를 조회합니다. 지금 로컬 Python에는 DB 드라이버가 설치되어 있지 않아 예제 셀을 기본적으로 건너뜁니다.


In [ ]:
demo_video_id = None  # DB가 연결되면 'dog_escape'로 변경
if demo_video_id:
    from db.behavior_events import get_behavior_events_for_aggregation
    timeline = get_behavior_events_for_aggregation(video_id=demo_video_id)
    matched = [event for event in timeline if event_matches_action(event, 'gate_climb')]
    episodes = count_episodes(matched)
    for episode in episodes:
        print(episode['video_id'], episode['start_time'], episode['end_time'], episode['event_ids'])
    print('영상에서 대조할 후보 구간:', len(episodes))
else:
    print('DB 연결을 설정하면 실제 이벤트를 조회할 수 있습니다.')


## 원본 영상 검토: dog_escape

이 영상의 길이는 약 56초입니다. 처음 약 36초에 울타리 장면이 나오고, 이후에는 도어벨 화면 등 다른 장면으로 바뀝니다. 아래 그림은 0~36초를 1초 간격으로 뽑은 샘플입니다. **약 6~13초와 25~31초는 검토 후보**이며, 실제 시작·종료와 별도 행동 여부는 영상 재생으로 확인해야 합니다. “울타리에 올라탐”은 앞발이나 몸을 울타리 위에 올린 동작으로 정의하고, 단순히 근처에 있는 장면은 제외합니다. 성공적으로 넘었는지는 이 질문에서 세지 않습니다.

![dog_escape 첫 36초의 1초 간격 샘플](assets/dog_escape_gate_review.jpg)


In [ ]:
from IPython.display import Video, display
video_files = sorted((Path.home() / 'Downloads').glob('chunks_dog_escape_*.mp4'))
print('찾은 원본 영상:', [path.name for path in video_files])
if video_files:
    display(Video(filename=str(video_files[0]), embed=True, width=640))
else:
    print('원본 영상이 없습니다. 로컬 영상 경로를 지정하세요.')


In [ ]:
# 원본 영상을 재생한 뒤 시작·끝이 확인된 구간만 기록하세요.
verified_intervals = [
    # {'start_time': 6.0, 'end_time': 13.0, 'action': 'gate_climb', 'note': '확인한 근거'},
]
print('사람이 검토한 구간 수:', len(verified_intervals))
